## Aims and Objectives of the project ##

The main goal is to use these features to build a Credit Scoring Model that can predict the probability of loan default.
This helps financial institutions evaluate credit risk and make informed lending decisions.

## Dataset Overview ##

The Credit Scoring Dataset is commonly used to assess the creditworthiness of individuals based on their demographic, financial, and behavioral information.
It contains customer records with multiple attributes that can influence loan repayment capability.

Source: [https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data]
Number of Instances: [32581]
Number of Attributes: [12 features including the target]  
**Key Features:**
1. person_age – Age of the applicant  
2. person_income – Annual income of the applicant  
3. person_home_ownership – Home ownership status (RENT, OWN, MORTGAGE, etc.)  
4. person_emp_length – Length of employment in years  
5. loan_intent – Purpose of the loan (PERSONAL, EDUCATION, MEDICAL, etc.)  
6. loan_grade – Credit grade assigned to the loan (A–G)  
7. loan_amnt – Amount of the loan requested  
8. loan_int_rate – Interest rate applied to the loan  
9. loan_status – Target variable (1 = default, 0 = fully paid)  
10. loan_percent_income – Ratio of loan amount to annual income(loan_amnt / person_income)  
11. cb_person_default_on_file – Whether the applicant has a history of default (Y--> didn't pay the loan /N-->clean record)    
12. cb_person_cred_hist_length – Length of the applicant’s credit history (in years)  

## Import Libraries ##

In [ ]:
# 1. to handle the data
import pandas as pd
import numpy as np

# to visualize the dataset
import seaborn as sns
import plotly.express as px
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import  accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_auc_score , roc_curve
import math
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

## load data ##

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'credit_risk_dataset.csv'

## Data Expolration ##

In [ ]:
print("\n1.data dimensions:")
print(f"number of rows: {df.shape[0]}")
print(f"number of columns: {df.shape[1]}")


In [ ]:
print("\n2.First 10 rows:")
df.head(10)

In [ ]:
print("\n3.Data types:\n")
df.info()

In [ ]:
print("\n4.Summary statistics:")
df.describe()

In [ ]:
print("\n5.missing values count:")
print(df.isnull().sum())
print(f"\ntotal missing values: {df.isnull().sum().sum()}")

## Data Processing ##

**Handle duplicates**

In [ ]:

# 1. Check for duplicate rows
print("\n Duplicates checking:")
print(f"Total number of rows: {len(df)}")
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# 2. Display duplicate rows 
if df.duplicated().sum() > 0:
    duplicates = df[df.duplicated(keep=False)]
    
    
    print("\n Remove duplicates")
    df_before = len(df)
    df = df.drop_duplicates(keep='first')  
    df_after = len(df)
    
    print(f"Removed {df_before - df_after} duplicate rows")
    print(f"Number of rows after removal: {df_after}")
    
   
    df = df.reset_index(drop=True)
    print("Index has been reset")
else:
    print("\n No duplicate rows found in the dataset!")
    print("  Data is clean and ready for processing.")



## Exploratory Data Analysis(EDA) ##

**Univariate Analysis**

In [ ]:
# Age Distribution

# MAX AND MIN AGE
max_ = df['person_age'].max()
min_ = df['person_age'].min()
print(f"maximum Age {max_}")
print(f"minimum Age {min_}")

# people with an age between x and y
def age_group(arr):
    lenarr = len(arr)
    for i in range(0,lenarr-1):
        next = arr[i]+1
        num_people = df['person_age'].between(next,arr[i+1]).sum()
        print(f'Age between {next} and {arr[i+1]}: Number of people {num_people}')
        
age_group([0 ,18, 26, 36, 46, 56, 66])

In [ ]:
# Distribution of Personal income

# max and min income
max_ = df['person_income'].max()
min_ = df['person_income'].min()

print(f"maximum Income {max_}")
print(f"minimum Income {min_}")

#people with an income between x and y
def income_group(arr):
    lenarr = len(arr)
    for i in range(0,lenarr-1):
        next = arr[i]+1
        num_people = df['person_income'].between(next,arr[i+1]).sum()
        print(f'Income between {next} and {arr[i+1]}: Number of people {num_people}')
        
income_group([0, 25000, 50000, 75000, 100000,float('inf')])


In [ ]:
# Distribution of Home Ownership

custom_palette = sns.color_palette("crest", n_colors=df["person_home_ownership"].nunique())

plt.figure(figsize=(6,6))
# Pie charts aren't natively in sns, so we'll use countplot + patch pie-like feel
ax = sns.countplot(
    x="person_home_ownership",
    data=df,
    palette=custom_palette,
    order=df["person_home_ownership"].value_counts().index
)

# Add value labels
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', 
                (p.get_x() + p.get_width()/2., p.get_height()), 
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')

# Style
ax.set_title("Distribution of Home Ownership", fontsize=16, pad=15)
ax.set_xlabel("Home Ownership Type", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
sns.despine()

plt.show()

Observations: Most of the People taking a loan doesn't own their own house

In [ ]:
# Loan Intent Distribution

loan_counts = df["loan_intent"].value_counts()

# Custom palette (similar to Mint)
colors = sns.color_palette("crest", n_colors=len(loan_counts))

# Pie chart
plt.figure(figsize=(7,7))
plt.pie(
    loan_counts.values,
    labels=loan_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    wedgeprops={'edgecolor':'gray', 'linewidth':1})

plt.title("Distribution of Loan Intent", fontsize=16, pad=15)
plt.show()

Observation: Most loans are taken for education and medical purposes, while categories like venture and personal use have fewer applicants. This suggests lending is often linked to essential or growth-oriented needs rather than discretionary spending

In [ ]:
# Frequency of Loan Grades 

grade_order = ["A", "B", "C", "D", "E", "F", "G"]
grade_counts = df["loan_grade"].value_counts().reindex(grade_order)

# Colors
colors = sns.color_palette("crest", n_colors=len(grade_counts))

plt.figure(figsize=(8,5))

# Stem lines
plt.stem(grade_counts.index, grade_counts.values, basefmt=" ", linefmt="gray")

# Circles
plt.scatter(grade_counts.index, grade_counts.values, s=120, c=colors, zorder=3)

# Styling
plt.title("Distribution of Loan Grades", fontsize=16, pad=15)
plt.xlabel("Loan Grade")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

Observation: Most loans are concentrated in the A and B grades, indicating a large share of borrowers have relatively strong creditworthiness. The lower grades (E–G) are rare, highlighting fewer high-risk applicants in the dataset.

In [ ]:
# Analysis of Borrowed Loan Amounts

max_loan_amount = df['loan_amnt'].max()
min_loan_amount = df['loan_amnt'].min()

print(f"maximum Loan Amount {max_loan_amount}")
print(f"minimum Loan Amount {min_loan_amount}")

# people with an income between x and y
def loan_amount_group(arr):
    lenarr = len(arr)
    for i in range(0,lenarr-1):
        next = arr[i]+1
        num_people = df['loan_amnt'].between(next,arr[i+1]).sum()
        print(f'Loan Amount between {next} and {arr[i+1]}: Number of people {num_people}')
        
loan_amount_group([0, 5000, 10000, 15000, float('inf')])

In [ ]:
# Spread of Loan Interest Rates

plt.figure(figsize=(8,5))
sns.histplot(df["loan_int_rate"], kde=True, color="teal", bins=30)
plt.title("Distribution of Loan Interest Rates", fontsize=16, pad=15)
plt.xlabel("Interest Rate (%)")
plt.ylabel("Frequency")
plt.show()

Observation: The distribution of loan interest rates is right-skewed, with most loans clustered around 8–15%. Very high interest rates are less frequent, indicating they are offered to a smaller set of higher-risk borrowers

In [ ]:
# Credit History Length Overview

fig=px.histogram(df, x = 'cb_person_cred_hist_length', text_auto = '.2f',template = 'presentation', title = 'person credit history length',color_discrete_sequence=px.colors.sequential.Mint)
fig.update_layout()
fig.show()


In [ ]:
df.nunique()


In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(col)
    print(df[col].unique())
    print()


In [ ]:
for col in df.select_dtypes(include="object").columns:
    df.loc[:, col] = df[col].astype("category")

df.loc[:, 'loan_status'] = df['loan_status'].astype("category")

In [ ]:
print(df['loan_int_rate'].isna().sum())
print(df[['person_emp_length','loan_int_rate']].isna().sum())


In [ ]:
from sklearn.impute import KNNImputer
import pandas as pd

df_plot = df.copy()

df_plot['loan_int_rate'] = pd.to_numeric(df_plot['loan_int_rate'], errors='coerce')
df_plot['person_emp_length'] = pd.to_numeric(df_plot['person_emp_length'], errors='coerce')

imputer_plot = KNNImputer(n_neighbors=5)
df_plot[['person_emp_length','loan_int_rate']] = imputer_plot.fit_transform(
    df_plot[['person_emp_length','loan_int_rate']]
)

print(df_plot[['person_emp_length','loan_int_rate']].isna().sum())


In [ ]:
#Home Ownership vs Loan Status 
palette = ['#c6dbef', '#f2b6c6']
sns.countplot(x='person_home_ownership', hue='loan_status', data=df, palette=palette) 
plt.title("Home Ownership vs Loan Status") 
plt.show()

Borrowers who rent their homes show a higher number of loan defaults compared to homeowners, while borrowers who own their homes tend to have lower default rates, indicating that housing stability is associated with lower credit risk.

In [ ]:
#Average Loan Percent Income by Loan Status
palette = ['#dcedc8', '#f8d7da']
sns.barplot(x='loan_status', y='loan_percent_income', data=df, estimator='mean',palette=palette) 
plt.title("Average Loan Percent Income by Loan Status")
plt.show()

Defaulted loans have a higher average loan-to-income ratio compared to non-defaulted loans, suggesting that a higher financial burden increases the likelihood of loan default.

## Outliers Detection ##

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

n_cols = 2  
n_rows = math.ceil(len(numerical_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(f"Outlier check: {col}")
    axes[i].set_xlabel("")


for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# we handled just featuer which need
print("Rows with age > 120 before cleaning:", df[df['person_age'] > 120].shape[0])
print("Rows with emp_length > 50 before cleaning:", df[df['person_emp_length'] > 50].shape[0])
print("Rows with income > 1,000,000 before cleaning:", df[df['person_income'] > 1000000].shape[0])


df = df[(df['person_age'] <= 120) & 
        (df['person_emp_length'] <= 50) & 
        (df['person_income'] <= 3000000)].copy()


print("-" * 30)
print("Rows with age > 120 after cleaning:", df[df['person_age'] > 120].shape[0])
print("Rows with emp_length > 50 after cleaning:", df[df['person_emp_length'] > 50].shape[0])
print("Rows with income > 1,000,000 after cleaning:", df[df['person_income'] > 1000000].shape[0])
numeric_cols = [
    'person_age', 'person_income', 'person_emp_length'
]
plt.figure(figsize=(16, 20))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(4, 2, i)
    sns.boxplot(x=df[col], color='#2b7bba')
    plt.title(f'After Cleaning: {col}', fontsize=14, fontweight='bold')
    plt.xlabel('') 

plt.tight_layout()
plt.show()

NOTE: We considered creating advanced engineered features but decided NOT to include them as we felt the existing features in the dataset were sufficient and adding complex features might not significantly improve the model performance. Below are the features we considered:

In [ ]:
#Feature Engineering Stage

# 1. Debt-to-Income Ratio: The basic metric for measuring financial capacity (loan amount relative to income)
# df_encoded['debt_to_income_ratio'] = df_encoded['loan_amnt'] / (df_encoded['person_income'] + 1)

# 2. Interest Burden: Measuring interest burden (the effect of interest rate and loan amount on income)
# df_encoded['interest_burden'] = (df_encoded['loan_amnt'] * df_encoded['loan_int_rate'] / 100) / (df_encoded['person_income'] + 1)

# 3. Loan-to-Credit-History Ratio: Loan-to-credit-history ratio (a large loan with a short history = high risk)
# df_encoded['loan_history_ratio'] = df_encoded['loan_amnt'] / (df_encoded['cb_person_cred_hist_length'] * 1000 + 1)


## Feature Encoding ##

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

binary_cols = ["cb_person_default_on_file", "loan_status"]  
encoders = {} 

for col in binary_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le 

multi_cols = ["person_home_ownership", "loan_intent", "loan_grade"]
df_encoded = pd.get_dummies(df_encoded, columns=multi_cols, drop_first=True, dtype=int)

print("Data format after encoding:", df_encoded.shape)
df_encoded.head()

## Split Data ##

In [ ]:
X = df_encoded.drop("loan_status", axis=1)  
y = df_encoded["loan_status"]               

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## Handle Missing Values ##

In [ ]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

X_train[['person_emp_length','loan_int_rate']] = imputer.fit_transform(
    X_train[['person_emp_length','loan_int_rate']]
)

X_test[['person_emp_length','loan_int_rate']] = imputer.transform(
    X_test[['person_emp_length','loan_int_rate']]
)

In [ ]:
sns.heatmap(
    X_train.isnull(),
    cbar=False,
    yticklabels=False,
    cmap='Pastel1'
)


In [ ]:
print("Missing values in X_train:")
print(X_train.isna().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum())


In [ ]:
fig = px.scatter(
    df_plot,
    x="person_income",
    y="loan_amnt",
    color="loan_status",
    animation_frame="loan_intent",
    size="loan_int_rate",
    hover_data=["person_age", "person_home_ownership", "loan_grade"],
    color_discrete_map={'Fully Paid': 'green', 'Charged Off': 'red'},
    title="Loan Amount vs. Income by Loan Intent (Animated by Loan Status)"
)
fig.show()


Normality Check of Numerical Variables

In [ ]:
# num_cols = [col for col in df.select_dtypes(include=['float64', 'int64']).columns if col != "loan_status"]
num_cols = [
    'person_age', 
    'person_income', 
    'person_emp_length', 
    'loan_amnt', 
    'loan_int_rate', 
    'loan_percent_income', 
    'cb_person_cred_hist_length'
]

n_cols = 2   
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()  

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")


for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Feature Scaling of Numerical Variables ##

In [ ]:
scaler = MinMaxScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

n_cols = 2  
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(X_train[col], kde=True, bins=30, ax=axes[i])
    
    axes[i].set_title(f"{col} after MinMax Scaling", fontsize=11)
    axes[i].set_xlabel("Scaled value (0 → 1)")
    axes[i].set_ylabel("Frequency")
    axes[i].set_xlim(0, 1)
    axes[i].grid(axis='y', alpha=0.3)


for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
# If you scale before splitting, the mean and standard deviation are  computed using the entire dataset, causing data leakage because information from the test set leaks into model training.

In [ ]:
#Correlation Matrix
corr_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix of Numerical Features", fontsize=14)
plt.xticks(rotation=45)
plt.show()

## Train Baseline Models & Predictions ##

In [ ]:

models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, random_state=42),
   
    "Random Forest": RandomForestClassifier(
        n_estimators=100, 
        max_depth=10,        
        min_samples_split=10, 
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,        
        learning_rate=0.1, 
        use_label_encoder=False, 
        eval_metric='logloss', 
        random_state=42
    )
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Store results
results = []
trained_models = {}

print("\n" + "─" * 80)
print("Training Progress:")
print("─" * 80)

for name, model in models.items():
    print(f"\n{name}:")
    
    
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    print(f"  → CV F1-Scores: {cv_scores}")
    print(f"  → Mean CV F1: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    results.append({
        'Model': name,
        'CV_F1_Mean': cv_scores.mean(),
        'CV_F1_Std': cv_scores.std(),
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'AUC': auc,
    })
    
    trained_models[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  → Test Metrics: Acc={accuracy:.3f} | Prec={precision:.3f} | Rec={recall:.3f} | F1={f1:.3f}")

# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False)

print("\n" + "═" * 80)
print(" " * 25 + "MODEL COMPARISON (SORTED BY F1)")
print("═" * 80)
print(results_df.to_string(index=False, float_format='%.4f'))

## Model Evaluation ##

1. Confusion Matrix

In [ ]:


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, data) in enumerate(trained_models.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Default', 'Default'])
    
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f"Confusion Matrix:\n{name}")
    axes[i].grid(False)

plt.tight_layout()
plt.show()

Interpretation of Confusion Matrix: XGBoost Model (Best Performer)
1. True Negatives (TN ≈ 4921):

Meaning: These are the reliable customers that the model correctly identified. They were granted loans and paid them back as expected. This represents the bank's "Safe Zone" and primary source of interest income.

2. True Positives (TP ≈ 993):

Meaning: These are the high-risk customers that the model successfully caught. By correctly predicting they would fail to pay, the bank avoided significant financial loss. This shows the model's strength in identifying actual risk.

3. False Positives (FP ≈ 22):

Meaning: "False Alarms." The model wrongly flagged these good customers as risky. While the bank loses the potential interest from these individuals, the number is very low in XGBoost, meaning the bank isn't turning away many good customers.

4. False Negatives (FN ≈ 369):

Meaning: "Missed Dangers." The model predicted these people would pay, but they actually defaulted. This is the most damaging error. However, notice that XGBoost significantly reduced this number compared to the baseline, providing much better protection for the bank's capital.

3. ROC Curve and AUC

In [ ]:
plt.figure(figsize=(10, 7))

for name, data in trained_models.items():
    if data['y_pred_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, data['y_pred_proba'])
        current_model_auc = roc_auc_score(y_test, data['y_pred_proba'])
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {current_model_auc:.4f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity/Recall)')
plt.title('Receiver Operating Characteristic (ROC) Comparison')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
best_model = trained_models['XGBoost']['model']

importances = best_model.feature_importances_
feature_names = X_train.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10), palette='magma')

plt.title('Top 10 Most Important Features - XGBoost Model', fontsize=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature Name', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

Best Model: XGBoost is the clear winner for this credit risk project.

Superior Discrimination (AUC = 0.948): It has an exceptional ability to distinguish between reliable and risky borrowers.

Effective Risk Detection (Recall = 0.72): It successfully identified over 74% of actual defaulters, significantly outperforming the Logistic Regression baseline (which only caught 54%).

High Reliability (Precision = 0.97): When it flags a customer as "High Risk," it is correct 97% of the time, preventing the bank from losing good customers unnecessarily.

 While the Logistic Regression baseline failed to provide enough protection, XGBoost offers the perfect balance between protecting the bank's capital (lowering bad debt) and maintaining profitability. It is the most robust tool for a real-world Credit Scoring system.


## 🔹 Cost-Benefit Analysis

We analyze the financial impact of classification decisions by assigning costs to False Positives and False Negatives.


In [ ]:

# Cost-Benefit Analysis
# Assumptions:
cost_fp = 500   # approving risky client
cost_fn = 200   # rejecting good client

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

total_cost = fp * cost_fp + fn * cost_fn
print("Total Cost:", total_cost)



## 🔹 Model Calibration

Calibration ensures predicted probabilities reflect true risk.


In [ ]:

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

prob_pos = model.predict_proba(X_test)[:,1]
brier = brier_score_loss(y_test, prob_pos)
print("Brier Score:", brier)

prob_true, prob_pred = calibration_curve(y_test, prob_pos, n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted probability")
plt.ylabel("True probability")
plt.title("Calibration Curve")
plt.show()



## 🔹 SHAP Feature Importance


In [ ]:

import shap

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test)



## 🔹 Feature Engineering


In [ ]:

# Example Feature Engineering
df['debt_to_income'] = df['total_debt'] / (df['annual_income'] + 1)



## 🔹 Hyperparameter Tuning using GridSearchCV


In [ ]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3,5,7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100,200]
}

grid = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
